# Parking Occupancy Forecasting Data Aggregation

This notebook processes a large parking occupancy forecasting dataset in 20,000-row chunks to reduce memory usage.

The data is cleaned and aggregated by mobile zone, weekday, and hour to create a compact dataset for later analysis and joining with Brisbane parking meter data.

In [1]:
import pandas as pd

FILE = "parking_occupancy_forecasting.csv"
CHUNK_SIZE = 20_000
GROUP_COLS = ["MOBILE_ZONE", "WEEKDAY", "HOUR"]

## 1. Inspect Source Data

A small sample is loaded first to inspect the dataset structure, column names, and data types before processing the full file.

In [2]:
sample = pd.read_csv(FILE, nrows=10)

sample

,MOBILE_ZONE,DATE,HOUR,OCCUPANCY_PRED
0,4000731,2026-01-18,3,0
1,4000731,2026-01-18,9,0
2,4000731,2026-01-18,12,0
3,4000731,2026-01-18,14,0
4,4000731,2026-01-18,17,0
5,4000731,2026-01-18,21,0
6,4000731,2026-01-18,23,0
7,4000731,2026-01-19,1,0
8,4000731,2026-01-19,3,0
9,4000731,2026-01-19,5,0


In [3]:
print(sample.columns)
print(sample.dtypes)

Index(['MOBILE_ZONE', 'DATE', 'HOUR', 'OCCUPANCY_PRED'], dtype='object')
MOBILE_ZONE        int64
DATE              object
HOUR               int64
OCCUPANCY_PRED     int64
dtype: object


## 2. Test Chunk Processing

The dataset is read in chunks of 20,000 rows to avoid loading the entire file into memory at once.

The first chunk is used to test the transformation and aggregation logic before applying it to the full dataset.

In [4]:
reader = pd.read_csv(
    FILE,
    chunksize=CHUNK_SIZE
)

In [5]:
first_chunk = next(reader)

print(first_chunk.shape)
first_chunk.head()

(20000, 4)


,MOBILE_ZONE,DATE,HOUR,OCCUPANCY_PRED
0,4000731,2026-01-18,3,0
1,4000731,2026-01-18,9,0
2,4000731,2026-01-18,12,0
3,4000731,2026-01-18,14,0
4,4000731,2026-01-18,17,0


## 3. Prepare Occupancy Data

The DATE column is converted to datetime format and used to derive the weekday.

According to the dataset metadata, an OCCUPANCY_PRED value of 0 indicates that occupancy information is unavailable. Only valid occupancy levels from 1 to 5 are therefore retained for aggregation.

In [6]:
first_chunk["DATE"] = pd.to_datetime(
    first_chunk["DATE"],
    errors="raise"
)

first_chunk["WEEKDAY"] = first_chunk["DATE"].dt.day_name()

first_chunk.head()

,MOBILE_ZONE,DATE,HOUR,OCCUPANCY_PRED,WEEKDAY
0,4000731,2026-01-18,3,0,Sunday
1,4000731,2026-01-18,9,0,Sunday
2,4000731,2026-01-18,12,0,Sunday
3,4000731,2026-01-18,14,0,Sunday
4,4000731,2026-01-18,17,0,Sunday


In [7]:
first_chunk["VALID_OCCUPANCY"] = first_chunk["OCCUPANCY_PRED"].where(
    first_chunk["OCCUPANCY_PRED"].between(1, 5)
)

first_chunk[
    ["OCCUPANCY_PRED", "VALID_OCCUPANCY"]
].head(20)

,OCCUPANCY_PRED,VALID_OCCUPANCY
0,0,NaN
1,0,NaN
2,0,NaN
3,0,NaN
4,0,NaN
5,0,NaN
6,0,NaN
7,0,NaN
8,0,NaN
9,0,NaN


## 4. Test Aggregation on the First Chunk

The first chunk is aggregated by MOBILE_ZONE, WEEKDAY, and HOUR.

For each group, the sum and count of valid occupancy observations are retained. These values are used later to calculate the overall weighted average across all chunks.

In [8]:
first_grouped = (
    first_chunk
    .groupby(GROUP_COLS)
    .agg(
        Occupancy_Sum=("VALID_OCCUPANCY", "sum"),
        Occupancy_Count=("VALID_OCCUPANCY", "count")
    )
    .reset_index()
)

first_grouped.head(20)

,MOBILE_ZONE,WEEKDAY,HOUR,Occupancy_Sum,Occupancy_Count
0,4000692,Friday,0,11.0,6
1,4000692,Friday,1,4.0,2
2,4000692,Friday,2,10.0,6
3,4000692,Friday,3,8.0,6
4,4000692,Friday,4,7.0,5
5,4000692,Friday,5,9.0,5
6,4000692,Friday,7,5.0,3
7,4000692,Friday,9,12.0,4
8,4000692,Friday,10,17.0,6
9,4000692,Friday,11,14.0,4


In [9]:
print("Before aggregation:", first_chunk.shape)
print("After aggregation:", first_grouped.shape)

Before aggregation: (20000, 6)
After aggregation: (5220, 5)


## 5. Process the Full Dataset

The same transformation and aggregation steps are applied to every 20,000-row chunk in the full dataset.

Each chunk produces a smaller summary table, which is stored for final consolidation.

In [10]:
results = []

for chunk in pd.read_csv(FILE, chunksize=CHUNK_SIZE):

    chunk["DATE"] = pd.to_datetime(
        chunk["DATE"],
        errors="raise"
    )

    chunk["WEEKDAY"] = chunk["DATE"].dt.day_name()

    chunk["VALID_OCCUPANCY"] = chunk["OCCUPANCY_PRED"].where(
        chunk["OCCUPANCY_PRED"].between(1, 5)
    )

    grouped = (
        chunk
        .groupby(GROUP_COLS)
        .agg(
            Occupancy_Sum=("VALID_OCCUPANCY", "sum"),
            Occupancy_Count=("VALID_OCCUPANCY", "count")
        )
        .reset_index()
    )

    results.append(grouped)

## 6. Combine Chunk-Level Results

All chunk-level summaries are concatenated into a single intermediate dataset.

In [11]:
combined = pd.concat(results, ignore_index=True)

print(combined.shape)
combined.head()

(633442, 5)


,MOBILE_ZONE,WEEKDAY,HOUR,Occupancy_Sum,Occupancy_Count
0,4000692,Friday,0,11.0,6
1,4000692,Friday,1,4.0,2
2,4000692,Friday,2,10.0,6
3,4000692,Friday,3,8.0,6
4,4000692,Friday,4,7.0,5


## 7. Final Aggregation

The combined chunk results are grouped again by MOBILE_ZONE, WEEKDAY, and HOUR.

This second aggregation is required because the same group may appear in multiple chunks. Occupancy sums and counts are therefore combined across all chunks.

In [12]:
final = (
    combined
    .groupby(GROUP_COLS)
    .agg(
        Occupancy_Sum=("Occupancy_Sum", "sum"),
        Occupancy_Count=("Occupancy_Count", "sum")
    )
    .reset_index()
)

print(final.shape)
final.head()

(94647, 5)


,MOBILE_ZONE,WEEKDAY,HOUR,Occupancy_Sum,Occupancy_Count
0,4000103,Friday,0,20.0,19
1,4000103,Friday,1,30.0,21
2,4000103,Friday,2,41.0,22
3,4000103,Friday,3,31.0,21
4,4000103,Friday,4,27.0,20


## 8. Calculate Average Occupancy Level

The final average occupancy level is calculated as:

**Total Occupancy Sum / Total Valid Observation Count**

Using the total sum and total count ensures that the average is correctly weighted across chunks.

In [13]:
final["Avg_Occupancy_Level"] = (
    final["Occupancy_Sum"] / final["Occupancy_Count"]
)

In [14]:
final.head(20)

,MOBILE_ZONE,WEEKDAY,HOUR,Occupancy_Sum,Occupancy_Count,Avg_Occupancy_Level
0,4000103,Friday,0,20.0,19,1.052632
1,4000103,Friday,1,30.0,21,1.428571
2,4000103,Friday,2,41.0,22,1.863636
3,4000103,Friday,3,31.0,21,1.476190
4,4000103,Friday,4,27.0,20,1.350000
5,4000103,Friday,5,40.0,23,1.739130
6,4000103,Friday,6,64.0,25,2.560000
7,4000103,Friday,7,89.0,26,3.423077
8,4000103,Friday,8,115.0,26,4.423077
9,4000103,Friday,9,124.0,26,4.769231


## 9. Validate the Aggregated Dataset

Groups with no valid occupancy observations are identified separately.

These groups represent combinations where all OCCUPANCY_PRED values were 0, meaning occupancy information was unavailable rather than zero occupancy.

In [15]:
zero_valid_groups = (final["Occupancy_Count"] == 0).sum()

print(
    "Groups with no valid occupancy observations:",
    zero_valid_groups
)

Groups with no valid occupancy observations: 8630


### Final Validation Checks

The final dataset is checked for:

- duplicate MOBILE_ZONE–WEEKDAY–HOUR combinations
- average occupancy levels outside the valid 1–5 range
- final dataset dimensions

In [16]:
print("Final shape:", final.shape)

print(
    final["Avg_Occupancy_Level"].describe()
)

print(
    "Duplicate groups:",
    final.duplicated(GROUP_COLS).sum()
)

valid_avg = final["Avg_Occupancy_Level"].dropna()

print(
    "All valid averages within 1–5:",
    valid_avg.between(1, 5).all()
)

Final shape: (94647, 6)
count    86017.000000
mean         2.251749
std          1.224376
min          1.000000
25%          1.090909
50%          1.900000
75%          3.083333
max          5.000000
Name: Avg_Occupancy_Level, dtype: float64
Duplicate groups: 0
All valid averages within 1–5: True


## 10. Export Aggregated Dataset

The validated aggregated dataset is exported as a CSV file for downstream analysis and joining with the Brisbane parking meter dataset.

In [17]:
final.to_csv(
    "parking_forecast_aggregated.csv",
    index=False
)

print("File exported successfully.")

File exported successfully.
